In [1]:
import numpy as np
import pandas as pd
import torch
import einops as eops

In [2]:
import torch.nn as nn 
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

In [ ]:
class HospitalSelectorModel(nn.Module):
    def __init__(self, _nFeature :int):
        super(HospitalSelectorModel, self).__init__()

        self._nFeature = _nFeature
        self.model =  nn.Sequential(
            nn.Linear(_nFeature, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
        
    def forward(self, inp :np.ndarray):
        return self.model(inp)

   

In [16]:
class Train():
    def __init__(self, _nFeature :int):
        self.model = HospitalSelectorModel(_nFeature)
        self.loss = nn.BCEWithLogitsLoss()
        self.optim = torch.optim.Adam(self.model.parameters(), lr=0.01)

    def train(self, X :np.ndarray, Y :np.ndarray, epochs:int=100, _nbatch:int=32):
        print("Training starts")
        model = self.model
        optim = self.optim
        loss_fn = self.loss

        Y = np.expand_dims(Y, -1)
        X = torch.from_numpy(X).float()
        Y = torch.from_numpy(Y).float()

        inp = TensorDataset(X, Y)
        print(type(inp), inp)
        data_loader = DataLoader(inp, batch_size=_nbatch, shuffle=True)
            
        for epoch in range(epochs):
            model.train()

            for i, (train_X, train_Y) in enumerate(data_loader):
                optim.zero_grad()
                logits = model(train_X)
                logit_loss = loss_fn(logits, train_Y)
                logit_loss.backward()
                optim.step()

            with torch.no_grad():
                idx = epoch%X.shape[0]
                pred = model(X[idx:idx+1])
                loss = loss_fn(pred, Y[idx:idx+1])
                print(f"Finishing epoch: {epoch} with {loss.item()*100}% logit loss")


    #assume inp: n-hospital with m-features each hospital -> min(top k-pred, #pred>=50%)
    def pred(self, inp :np.ndarray, k :int=3):
        with torch.no_grad():
            inp = torch.Tensor(inp)
            pred_score = torch.sigmoid(self.model(inp))
            idx = []
            print(pred_score)
            for pred in pred_score:
                arg = torch.argmax(pred_score)
                if pred_score[arg]<0.8:
                    break
                pred_score[arg] = 0
                idx.append(arg)
                
            l = len(idx)
            return idx[:min(l, k)] 

    def save(self, name :str):
        torch.save(self.model, name)

In [17]:
data = pd.read_csv("Data.csv")

data


,DISTANCE,GOVT,PVT,BUDGET,TIME,CRITICALITY,GO
0,2.5,1,0,500,10,0,1
1,3.0,1,0,300,12,1,1
2,1.5,1,0,150,5,2,1
3,4.0,1,0,0,15,3,1
4,2.0,0,1,400,8,0,0
...,...,...,...,...,...,...,...
195,62.0,0,1,90000,105,1,1
196,68.0,0,1,120000,110,3,0
197,100.0,1,0,3000,160,2,0
198,95.0,1,0,2500,155,1,0


In [18]:

y = data['GO']
x = data.drop('GO', axis=1)
continuous_cols = ['DISTANCE', 'BUDGET', 'TIME', 'CRITICALITY']

scaler = StandardScaler()
x[continuous_cols] = scaler.fit_transform(x[continuous_cols])


y = y.to_numpy()
x = x.to_numpy()
x

array([[-0.91167467,  1.        ,  0.        , -0.65579142, -0.8826028 ,
        -1.30618272],
       [-0.89322718,  1.        ,  0.        , -0.66051096, -0.83449144,
        -0.4688861 ],
       [-0.94856963,  1.        ,  0.        , -0.66405061, -1.00288118,
         0.36841051],
       ...,
       [ 2.68558474,  1.        ,  0.        , -0.59679721,  2.72574879,
         0.36841051],
       [ 2.5011099 ,  1.        ,  0.        , -0.60859605,  2.6054704 ,
        -0.4688861 ],
       [-0.76409479,  1.        ,  0.        , -0.66759026, -0.83449144,
         1.20570712]], shape=(200, 6))

In [19]:
model = Train(6)
model.train(x, y, _nbatch=32, epochs=128)

Training starts
<class 'torch.utils.data.dataset.TensorDataset'> <torch.utils.data.dataset.TensorDataset object at 0x000001692BC19950>
Finishing epoch: 0 with 15.961091220378876% logit loss
Finishing epoch: 1 with 6.28354474902153% logit loss
Finishing epoch: 2 with 3.9916370064020157% logit loss
Finishing epoch: 3 with 6.7882053554058075% logit loss
Finishing epoch: 4 with 99.75262880325317% logit loss
Finishing epoch: 5 with 11.589907109737396% logit loss
Finishing epoch: 6 with 15.08864164352417% logit loss
Finishing epoch: 7 with 0.6737283430993557% logit loss
Finishing epoch: 8 with 0.2749499399214983% logit loss
Finishing epoch: 9 with 1.6294436529278755% logit loss
Finishing epoch: 10 with 5.618929862976074% logit loss
Finishing epoch: 11 with 2.3217443376779556% logit loss
Finishing epoch: 12 with 0.38401607889682055% logit loss
Finishing epoch: 13 with 2.5660037994384766% logit loss
Finishing epoch: 14 with 3.319096565246582% logit loss
Finishing epoch: 15 with 1.0595608502626

In [20]:
preds = model.pred(x[0:20, :], 20)
for pred in preds:
    print(True if y[pred.item()]==1 else False)

tensor([[1.0000e+00],
        [1.0000e+00],
        [1.0000e+00],
        [1.0000e+00],
        [1.4530e-02],
        [1.0000e+00],
        [1.0033e-03],
        [1.0000e+00],
        [1.0000e+00],
        [9.9998e-01],
        [2.6929e-03],
        [1.0000e+00],
        [9.9995e-01],
        [5.8531e-07],
        [5.1893e-04],
        [1.0000e+00],
        [6.1769e-03],
        [5.4607e-07],
        [9.6621e-01],
        [1.0000e+00]])
True
True
True
True
True
True
True
True
True
True
True
True
True


In [21]:
y[0:20]

array([1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1])

In [22]:
model.save("HospSelectorModel.h5")

AttributeError: 'HospitalSelectorModel' object has no attribute 'save'